In [ ]:
from py123d.api import SceneAPI, SceneFilter, get_filtered_scenes

scene_filter = SceneFilter(
    datasets=["nuplan-mini"],
    split_names=None,
    log_names=None,
    # scene_uuids=[
    #     "28a0f85d-2eaf-5e76-966b-6c2dc1dcb11e",
    #     "cdb256f4-4be5-56df-8b3b-447a49b00f2d",
    #     "2685645c-2a31-552b-a1cf-9dd1eb6de030",
    #     "6586443b-a05c-55f1-8513-bd8a78436889",
    #     "7ff3b3a2-555f-59aa-9f49-36823b7e308f",
    #     "184c7ef9-2452-58b8-99c8-574f196544ed",
    # ],
    target_iteration_duration_s=0.1,  # 10Hz iteration frequency
    future_duration_s=10.0,  # Look up to 1 second into the future.
    history_duration_s=0,  # Look up to 0.5 seconds into the past.
    # required_scene_modalities=["ego_state_se3", "lidar.lidar_merged"],
    required_scene_modalities=["ego_state_se3"],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from nav123d.geometry.trajectory import TrajectorySampling
from py123d.visualization.matplotlib.observation import add_scene_on_ax

from nav123d.agents.constant_velocity_agent import ConstantVelocityAgent
from nav123d.api import scene_api_to_agent_api
from nav123d.metrics.pdm_metric import PDMMetric, _resample_trajectory_se2

scene: SceneAPI = np.random.choice(scenes)  # type: ignore

ego_state_se2 = scene.get_ego_state_se3_at_iteration(0).ego_state_se2  # type: ignore

fig, ax = plt.subplots(figsize=(10, 10))
add_scene_on_ax(ax, scene)


agent = ConstantVelocityAgent()
agent.initialize()
agent_api = scene_api_to_agent_api(scene, observation_type=agent.get_observation_type())
trajectory = agent.compute_trajectory(agent_api)

trajectory_abs = _resample_trajectory_se2(
    trajectory=trajectory,
    sampling=TrajectorySampling(time_horizon=4, interval_length=0.1),
    initial_ego_state_se2=ego_state_se2,
    convert_to_absolute=True,
)

print(trajectory_abs.pose_se2_array.shape)

ax.plot(
    trajectory_abs.pose_se2_array[:, 0],
    trajectory_abs.pose_se2_array[:, 1],
    marker="o",
    label="Predicted Trajectory",
    zorder=5,
)


pdm_metric = PDMMetric()
df = pdm_metric.compute_metric(scene, agent_trajectory=trajectory)

# df.iloc[0].to_dict()
df